# Pipeline di Estrazione e Preprocessing Dati - Head & Neck Cancer

Questo notebook documenta la complessa pipeline di ingegnerizzazione dei dati che trasforma i raw data disaggregati (clinici, di follow-up e fattori di rischio) in un dataset strutturato e pronto per l'addestramento dei modelli di Machine Learning per l'analisi di sopravvivenza.



---

### Fase 1: Caricamento Dati Raw
Il processo inizia importando i tre file Excel originali contenenti le informazioni disgiunte del progetto TCGA-HNSC:
* **Dati Clinici:** Informazioni anagrafiche, diagnosi e trattamenti.
* **Follow-up:** Tempistiche di contatto e stato in vita.
* **Fattori di Rischio:** Dati relativi all'esposizione a tabacco e alcol.
Tutti gli ID dei pazienti (`case_id`) vengono normalizzati (spazi rimossi, convertiti in maiuscolo) per garantire un matching perfetto nelle fasi successive.

### Fase 2: Aggregazione Dati Clinici per Paziente
Poiché un paziente può avere record multipli, i dati vengono aggregati logicamente:
* **Stadiazione TNM:** Viene data sempre priorità allo stadio *patologico* (più accurato post-chirurgia); in sua assenza o in caso di incompletezza (es. valori TX, NX), il sistema fa un fallback automatico sul TNM *clinico*.
* **Trattamenti (Trt):** Vengono raggruppati in macro-categorie binarie (Farmacologico, Radioterapico, Chirurgico), considerando solo i trattamenti effettivamente somministrati (`yes`).
* **Intento Clinico (Intent):** Vengono codificate le finalità terapeutiche (Curativo, Adiuvante, Palliativo, ecc.) scartando i record dei pazienti che hanno esplicitamente rifiutato le cure.

### Fase 3: Follow-up (Filtro Last Contact)
Essendo il nostro obiettivo l'analisi di sopravvivenza, la variabile temporale è cruciale. Filtriamo il dataframe di follow-up isolando esclusivamente i record etichettati come "Last Contact", estraendo così il numero definitivo di giorni di osservazione validi per ciascun paziente.

### Fase 4: Fattori di Rischio e Gestione Missing
* I pazienti esposti allo *Smokeless Tobacco* vengono esclusi dal dataset, poiché presentano un profilo oncologico e di rischio non comparabile al resto della coorte.
* **Imputazione Deduttiva:** Per i pazienti classificati come "Lifelong Non-Smoker", il valore dei *pack years* viene forzato a `0.0` (deduzione logica inconfutabile).
* **Missing Value Preservation:** Tutti gli altri valori numerici mancanti (es. giorni di alcol a settimana) vengono deliberatamente lasciati come `NaN`. Non viene più imposta una media o mediana "piatta", delegando l'imputazione avanzata all'algoritmo multivariato MICE nelle pipeline dei modelli a valle.

### Fase 5: Merge Globale ed Encoding
I tre set di dati aggregati vengono uniti in un unico Dataframe. Segue una massiccia fase di Feature Engineering:
* **Codifica Binaria/One-Hot:** Variabili come Sede del Tumore, Sesso e Lateralità vengono trasformate in feature numeriche adatte ai modelli ML.
* **Smoking Status Ordinale:** Lo status di fumatore viene mappato su una scala ordinale che riflette la gravità e la durata dell'esposizione (da Non-fumatore a Fumatore Attivo).
* **Encoding TNM Granulare:** I codici TNM vengono convertiti in valori ordinali numerici, prestando molta attenzione a differenziare stadi clinicamente distinti (ad esempio, T4A, che è resecabile, mantiene un peso diverso rispetto a T4B, che è inoperabile). I valori mancanti o ambigui vengono etichettati con `-1`.

### Fase 6: Pulizia Finale e Output
Vengono rimosse le colonne di servizio ormai inutili (come l'ID del paziente). Il sistema esegue un ultimo controllo di qualità eliminando gli eventuali pazienti rimasti privi di un dato temporale di follow-up valido. Il dataset finale pulito viene esportato in formato CSV, pronto per l'addestramento algoritmico.

In [ ]:
import pandas as pd
import numpy as np
import math
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURAZIONE
# ─────────────────────────────────────────────────────────────────────────────
FILE_ORAL = "oral_cancer_dataset.xlsx"
FILE_FOLLOW = "follow_up.xlsx"
FILE_RISK = "fattori_di_rischio.xlsx"
FILE_OUTPUT = "dataset.csv"

ID = 'cases.case_id'

print("═" * 65)
print("  PIPELINE FROM RAW — oral cancer TCGA-HNSC")
print("═" * 65)

# ─────────────────────────────────────────────────────────────────────────────
# 1. CARICAMENTO
# ─────────────────────────────────────────────────────────────────────────────
print("\n[1/6] Caricamento file raw...")

oral = pd.read_excel(FILE_ORAL)
follow = pd.read_excel(FILE_FOLLOW)
risk = pd.read_excel(FILE_RISK)

for df in [oral, follow, risk]:
    df[ID] = df[ID].astype(str).str.strip().str.upper()

print(f"    Clinico: {oral.shape} | Follow-up: {follow.shape} | Rischio: {risk.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. AGGREGAZIONE DATI CLINICI PER PAZIENTE
# ─────────────────────────────────────────────────────────────────────────────
print("\n[2/6] Aggregazione per paziente...")

EXCL_GEN = {"'--", '--', 'nan', 'none', 'null', '', 'not reported', 'unknown'}
EXCL_TNM = EXCL_GEN | {'tx', 'nx', 'mx'}

def first_valid(series, excl=None):
    if excl is None:
        excl = EXCL_GEN
    vals = series.dropna().astype(str)
    vals = vals[~vals.str.strip().str.lower().isin(excl)]
    return vals.iloc[0].strip().upper() if not vals.empty else None

def get_tnm(group, path_col, clin_col):
    return first_valid(group[path_col], EXCL_TNM) or \
        first_valid(group[clin_col], EXCL_TNM)

PHARMA_TYPES = {
    'Chemotherapy', 'Pharmaceutical Therapy, NOS',
    'Immunotherapy (Including Vaccines)', 'Targeted Molecular Therapy'
}
RAD_TYPES = {
    'Radiation Therapy, NOS', 'Radiation, Cyberknife',
    'Radiation, External Beam', 'Radiation, Implants',
    'Radiation, Stereotactic/Gamma Knife/SRS'
}
SURG_TYPES = {'Surgery, NOS'}

EXCL_TRT = {"'--", '--', 'nan', 'none', 'null', '', 'not reported', 'unknown'}

agg_rows = []

for case_id, group in oral.groupby(ID, sort=True):
    primary_site = first_valid(group['cases.primary_site'])
    age = first_valid(group['demographic.age_at_index'])
    ethnicity = first_valid(group['demographic.ethnicity'])
    gender = first_valid(group['gender'])
    race = first_valid(group['demographic.race'])
    vital = first_valid(group['demographic.vital_status'])
    laterality = first_valid(group['diagnoses.laterality'])

    final_t = get_tnm(group, 'diagnoses.ajcc_pathologic_t', 'diagnoses.ajcc_clinical_t')
    final_n = get_tnm(group, 'diagnoses.ajcc_pathologic_n', 'diagnoses.ajcc_clinical_n')
    final_m = get_tnm(group, 'diagnoses.ajcc_pathologic_m', 'diagnoses.ajcc_clinical_m')
    final_stage = get_tnm(group, 'diagnoses.ajcc_pathologic_stage', 'diagnoses.ajcc_clinical_stage')

    class_vals = set(group['diagnoses.classification_of_tumor'].dropna().astype(str)) - EXCL_TRT

    grp_yes = group[group['treatments.treatment_or_therapy'].astype(str).str.lower() == 'yes']
    trt_vals = set(grp_yes['treatments.treatment_type'].dropna().astype(str)) - EXCL_TRT

    grp_active = group[group['treatments.treatment_or_therapy'].astype(str).str.lower()
    .isin(['yes', 'unknown'])]
    intent_vals = set(grp_active['treatments.treatment_intent_type'].dropna().astype(str))
    intent_vals = {v for v in intent_vals if v.lower().strip() not in EXCL_TRT}

    agg_rows.append({
        ID: case_id,
        'cases.primary_site': primary_site,
        'demographic.age_at_index': age,
        'demographic.ethnicity': ethnicity,
        'gender': gender,
        'demographic.race': race,
        'demographic.vital_status': vital,
        'diagnoses.laterality': laterality,
        'Final_T': final_t,
        'Final_N': final_n,
        'Final_M': final_m,
        'Final_Stage': final_stage,
        'Hist_METASTASIS': int('metastasis' in class_vals),
        'Hist_RECURRENCE': int('recurrence' in class_vals),
        'Hist_FIELD CANCER': int(any(v in class_vals for v in
                                     ['Prior primary', 'Subsequent Primary', 'Synchronous primary'])),
        'Trt_PHARMACEUTICAL': int(bool(PHARMA_TYPES & trt_vals)),
        'Trt_RADIATION': int(bool(RAD_TYPES & trt_vals)),
        'Trt_SURGERY': int('Surgery, NOS' in trt_vals),
        'Intent_ADJUVANT': int('Adjuvant' in intent_vals),
        'Intent_CURE': int('Cure' in intent_vals),
        'Intent_BEFORETRT': int(bool({'Induction', 'Neoadjuvant'} & intent_vals)),
        'Intent_PALLIATIVE': int('Palliative' in intent_vals),
    })

df = pd.DataFrame(agg_rows)

print(f"    Pazienti aggregati: {len(df)}")
print(f"    Trt:    PHARMA={df['Trt_PHARMACEUTICAL'].sum()}, RAD={df['Trt_RADIATION'].sum()}, SURG={df['Trt_SURGERY'].sum()}")
print(f"    Intent: ADJ={df['Intent_ADJUVANT'].sum()}, CURE={df['Intent_CURE'].sum()}, PALL={df['Intent_PALLIATIVE'].sum()}, BEFORETRT={df['Intent_BEFORETRT'].sum()}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. FOLLOW-UP (filtro Last Contact)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[3/6] Follow-up: filtro Last Contact...")

df_fup = follow.copy()
df_fup['follow_ups.timepoint_category'] = (
    df_fup['follow_ups.timepoint_category'].astype(str).str.strip().str.title()
)
df_fup_last = df_fup[df_fup['follow_ups.timepoint_category'] == 'Last Contact'].copy()
df_fup_last['follow_ups.days_to_follow_up'] = pd.to_numeric(
    df_fup_last['follow_ups.days_to_follow_up'], errors='coerce'
)
df_fup_clean = (
    df_fup_last.groupby(ID)['follow_ups.days_to_follow_up']
    .first().reset_index()
)
print(f"    Pazienti con Last Contact: {len(df_fup_clean)}")

# ─────────────────────────────────────────────────────────────────────────────
# 4. FATTORI DI RISCHIO
# ─────────────────────────────────────────────────────────────────────────────
print("\n[4/6] Fattori di rischio...")

smokeless_ids = set(risk[risk['exposures.exposure_type'] == 'Smokeless Tobacco'][ID])
risk_c = risk[~risk[ID].isin(smokeless_ids)].copy()
print(f"    Rimossi {len(smokeless_ids)} pazienti Smokeless Tobacco → {risk_c[ID].nunique()} rimasti")

risk_c = risk_c.replace(["'--", 'Not Reported', 'Unknown', 'unknown'], np.nan)
COLS_NUM = [
    'exposures.alcohol_days_per_week',
    'exposures.alcohol_drinks_per_day',
    'exposures.pack_years_smoked'
]
for col in COLS_NUM:
    risk_c[col] = pd.to_numeric(risk_c[col], errors='coerce')

risk_clean = (
    risk_c[[ID, 'exposures.tobacco_smoking_status'] + COLS_NUM]
    .groupby(ID).first().reset_index()
)

risk_clean.loc[
    risk_clean['exposures.tobacco_smoking_status'] == 'Lifelong Non-Smoker',
    'exposures.pack_years_smoked'
] = 0.0

print(f"    Valori numerici mancanti preservati come NaN per imputazione a valle.")

# ─────────────────────────────────────────────────────────────────────────────
# 5. MERGE GLOBALE E ENCODING
# ─────────────────────────────────────────────────────────────────────────────
print("\n[5/6] Merge e feature encoding...")

df = pd.merge(df, df_fup_clean, on=ID, how='left')
df = pd.merge(df, risk_clean, on=ID, how='left')

STR_COLS = [
    'demographic.vital_status', 'gender', 'cases.primary_site',
    'demographic.race', 'demographic.ethnicity', 'diagnoses.laterality',
    'Final_T', 'Final_N', 'Final_M', 'exposures.tobacco_smoking_status'
]
for col in STR_COLS:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()
        df[col] = df[col].replace(['NAN', 'NONE', 'NULL', 'N/A', 'NA', "'--", '--'], np.nan)

df['demographic.vital_status_DEAD'] = (df['demographic.vital_status'] == 'DEAD').astype(int)
df['gender_MALE'] = (df['gender'] == 'MALE').astype(int)
df = df.drop(columns=['demographic.vital_status', 'gender'])

def group_site(val):
    s = str(val)
    if 'FLOOR OF MOUTH' in s: return 'FLOOR OF MOUTH'
    if 'TONGUE' in s: return 'TONGUE'
    if 'LARYNX' in s or 'HYPOPHARYNX' in s: return 'LARYNX & HYPOPHARYNX'
    if 'TONSIL' in s or 'OROPHARYNX' in s: return 'OROPHARYNX & TONSIL'
    if 'LIP' in s: return 'LIP'
    return 'OTHER ORAL CAVITY'

df['cases.primary_site'] = df['cases.primary_site'].apply(group_site)
dummies_site = pd.get_dummies(
    df['cases.primary_site'], prefix='cases.primary_site', dummy_na=False, dtype=int
)
df = pd.concat([df.drop(columns=['cases.primary_site']), dummies_site], axis=1)

def group_race(val):
    if pd.isna(val): return np.nan
    s = str(val)
    if 'WHITE' in s: return 'WHITE'
    if 'ASIAN' in s: return 'ASIAN'
    return 'NOT WHITE'

df['demographic.race'] = df['demographic.race'].apply(group_race)

SMOKING_ORDER = {
    'LIFELONG NON-SMOKER': 0,
    'CURRENT REFORMED SMOKER FOR > 15 YRS': 1,
    'CURRENT REFORMED SMOKER FOR < OR = 15 YRS': 2,
    'CURRENT SMOKER': 3,
}
df['exposures.tobacco_smoking_status_enc'] = (
    df['exposures.tobacco_smoking_status']
    .map(SMOKING_ORDER)
    .fillna(-1).astype(int)
)
df = df.drop(columns=['exposures.tobacco_smoking_status'])

T_MAP = {'TIS': 0, 'T0': 1, 'T1': 2, 'T2': 3, 'T3': 4, 'T4': 5, 'T4A': 6, 'T4B': 7}
N_MAP = {'N0': 0, 'N1': 1, 'N1B': 2, 'N2': 3, 'N2A': 4, 'N2B': 5, 'N2C': 6, 'N3': 7}
M_MAP = {'M0': 0, 'M1': 1}

df['Final_T_enc'] = df['Final_T'].map(T_MAP).fillna(-1).astype(int)
df['Final_N_enc'] = df['Final_N'].map(N_MAP).fillna(-1).astype(int)
df['Final_M_enc'] = df['Final_M'].map(M_MAP).fillna(-1).astype(int)
df = df.drop(columns=['Final_T', 'Final_N', 'Final_M', 'Final_Stage'])

NOMINAL_COLS = ['diagnoses.laterality', 'demographic.race', 'demographic.ethnicity']
df = pd.get_dummies(df, columns=NOMINAL_COLS, dummy_na=True, dtype=int)
df.columns = df.columns.str.replace('_nan', '_Missing', regex=False)

# ─────────────────────────────────────────────────────────────────────────────
# 6. PULIZIA FINALE E OUTPUT
# ─────────────────────────────────────────────────────────────────────────────
print("\n[6/6] Pulizia finale e salvataggio...")

df = df.drop(columns=[ID, 'follow_ups.disease_response'], errors='ignore')

n_before = len(df)
df = df[df['follow_ups.days_to_follow_up'].notna()].reset_index(drop=True)
print(f"    Droppati {n_before - len(df)} paziente/i senza follow-up")

print(f"\n{'═' * 65}")
print(f"  COMPLETATO — {df.shape[0]} pazienti × {df.shape[1]} feature")
print(f"{'═' * 65}")
print(f"  Colonne: {df.columns.tolist()}")

df.to_csv(FILE_OUTPUT, index=False)
print(f"\n  Salvato in: {FILE_OUTPUT}")